In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
import json

from utils_wind_analysis import get_wind_direction_in_degrees, get_wind_speed

In [ ]:
base_folder = r'/home/leroquan@eawag.wroot.emp-eaw.ch/work_space/lucerne_100m_2025'

In [ ]:
grid_folder = os.path.join(base_folder, 'grid')
with open(os.path.join(grid_folder, 'parameters.json'), 'r') as file:
    grid_angle = json.load(file)["rotation"]

In [ ]:
bin_folder = os.path.join(base_folder, 'binary_data')

In [ ]:
base_output_folder = r"/storage/alplakes_test/lucerne_100m_2025"
output_folder = os.path.join(base_output_folder, 'wind_analysis')
os.makedirs(output_folder, exist_ok=True)

# Import xarray wind

In [ ]:
ds = xr.open_dataset(os.path.join(bin_folder, 'wind_with_vorticity.nc'))

# Define zones

In [ ]:
zone_map = {
    0: "alt",
    1: "ges",
    2: "luz",
    3: "alpnachsee",
}
zone_coord = {'luz':[[(0,100),(0,180)],
                    [(100,120),(85,130)],
                    [(120,145),(91,130)],
                    [(145,150),(95,130)]],
             'ges':[[(100,230),(0,180)]],
             'alt':[[(230,288),(0,180)]],
             'alpnachsee':[[(0,52),(0,38)]]}

In [ ]:
arr_zones = np.zeros((int(len(ds.y)),int(len(ds.x))))
for idx, sta_name in zone_map.items():
    for zones in zone_coord[sta_name]:
        arr_zones[zones[1][0]:zones[1][1],zones[0][0]:zones[0][1]] = idx

In [ ]:
plt.imshow(ds.speed.isel(time=0).values, origin='lower')
plt.imshow(arr_zones, origin='lower', alpha=0.5)
plt.colorbar()

# Statistics per zone

In [ ]:
def compute_mean_characteristics(ds, zone_name):
    u_mean = ds.u10.mean(dim=["x", "y"])
    v_mean = ds.v10.mean(dim=["x", "y"])
    speed_mean = ds.speed.mean(dim=["x", "y"])
    vorticity_mean = ds.vorticity.mean(dim=["x", "y"])
    vorticity_strength = np.abs(ds.vorticity).mean(dim=["x", "y"])

    direction_mean = get_wind_direction_in_degrees(
        u_mean, v_mean, grid_angle
    )
    direction_mean.name = "direction"

    # add zone dimension
    u_mean = u_mean.expand_dims(zone=[zone_name])
    v_mean = v_mean.expand_dims(zone=[zone_name])
    speed_mean = speed_mean.expand_dims(zone=[zone_name])
    direction_mean = direction_mean.expand_dims(zone=[zone_name])
    vorticity_mean = vorticity_mean.expand_dims(zone=[zone_name])
    vorticity_strength = vorticity_strength.expand_dims(zone=[zone_name])

    ds_mean = xr.Dataset(
        data_vars={
            "u10": u_mean,
            "v10": v_mean,
            "speed": speed_mean,
            "direction": direction_mean,
            "vorticity_mean": vorticity_mean,
            "vorticity_strength": vorticity_strength,
        }
    )

    return ds_mean

In [ ]:
ds_zone_list = []
ds_zone_list.append(compute_mean_characteristics(ds, 'all')) # First the entire lake

for zone_id, zone_name in zone_map.items():
    ds_zone = ds.where(arr_zones == zone_id)
    ds_mean_zone = compute_mean_characteristics(ds_zone, zone_name)
    ds_zone_list.append(ds_mean_zone)

In [ ]:
# Concat ds
ds_mean_all = xr.concat(ds_zone_list, dim="zone")

In [ ]:
ds_mean_all.to_netcdf(os.path.join(output_folder, "wind_stats_per_zone.nc"))

## Speed

In [ ]:
for idx, sta_name in zone_map.items():
    ds_mean_all.speed.isel(zone=idx).plot(label=sta_name)
plt.legend()

## Vorticity

In [ ]:
for idx, sta_name in zone_map.items():
    ds_mean_all.vorticity_mean.isel(zone=idx).plot(label=sta_name)
plt.legend()

In [ ]:
plt.figure(figsize=(15,7))
sta_name = 'luz'
ds_mean_all.vorticity_mean.sel(zone=sta_name).plot(label='Mean')
ds_mean_all.vorticity_strength.sel(zone=sta_name).plot(label='Strength')
plt.legend()
plt.title(f'Zone = {sta_name}')